# FactorVAE on the Point-in-Time S&P 500

**FactorVAE** (Duan et al., AAAI 2022) applied to my data. See the vault notes "Notes - FactorVAE ..." (the paper) and "Feature Table - Stock-Day Inputs for FactorVAE" (the inputs).

**The model in one paragraph.** Each stock's last 20 days of features go through a small recurrent network (GRU) that summarizes them. A **teacher** (encoder), used only in training, sees **tomorrow's actual returns** and finds a few hidden "factors" that explain them (the **posterior**). A **student** (predictor) sees **only the past** and learns to guess those factors (the **prior**). The **decoder** turns factors into a bell curve for each stock's return: mean = stock-specific part + exposures × factors. Predictions use only the student and the decoder, so no future information is used.

| | Setting |
|---|---|
| **Universe** | every S&P 500 member on each day (about 500 stocks; point-in-time) |
| **Inputs** | 25 features from `features_daily.csv.gz` (returns, risk, trading, news, earnings, fundamentals), last 20 days |
| **Target** | next day's excess return (vs the S&P 500 members' average), standardized across stocks each day |
| **Walk-forward** | re-trained **every year** on up to the past 5 years (the last year is used for early stopping); predicts every day of the next year. 2006–2022 |
| **Model settings** | fixed in advance (not tuned): 32 hidden units, 6 factors, 24 portfolios, KL weight 1 |
| **Evaluation** | 2006–2020 walk-forward and **2021–2022 validation**; test years (2023+) are not used |

**Benchmarks:** a linear model on the same 25 features, daily and weekly reversal, 12-month momentum, and the GDBN (on its own 52-stock universe, Section 6).

Training takes about **30 minutes** the first time. The predictions are then saved, so later runs load them in seconds (set `RETRAIN = True` to train again).

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as Fn
from sklearn.linear_model import Ridge

import pit_common as pc
from gdbn_utils import RESULTS

FEATS = ["exret_1d", "exret_5d", "exret_20d", "mom_12_1", "overnight", "intraday",
         "vol_1d", "range", "rvol_20", "beta_60", "ivol_60",
         "volume", "turnover_20", "amihud_20", "size",
         "sentiment", "has_news", "attention", "surprise",
         "days_since_earn", "sue", "bm", "gp", "lev", "ag"]
T_DAYS = 20                     # days of history per stock
H, K, M_PORT, GAMMA = 32, 6, 24, 1.0   # hidden units, factors, portfolios, KL weight (fixed in advance)
LR, MAX_EPOCHS, PATIENCE = 1e-3, 10, 2
WINDOW_YEARS, VAL_DAYS = 5, 252  # training window (max), last year of it for early stopping
YEARS = range(2006, 2023)        # predicted years
SEED = 0
RETRAIN = False                  # True: train again even if saved predictions exist
OUT_DIR = RESULTS
SCORES_FILE = OUT_DIR / "factorvae_scores.csv.gz"
torch.set_num_threads(8)
DEVICE = "cpu"                   # faster than the Mac GPU for a model this small

D = pc.load()                    # returns for evaluation (everything after 2022 blanked)
t0 = time.time()
df = pd.read_csv(pc.BUILT / "features_daily.csv.gz", usecols=["date", "permno", "in_sp500", "target_next"] + FEATS,
                 parse_dates=["date"], dtype={f: "float32" for f in FEATS + ["target_next"]})
df = df[df["date"] <= pc.EVAL_END]
DAYS = pd.DatetimeIndex(np.sort(df["date"].unique()))
PERM = np.sort(df["permno"].unique())
di, pj = DAYS.get_indexer(df["date"]), pd.Index(PERM).get_indexer(df["permno"])
X = np.full((len(DAYS), len(PERM), len(FEATS)), np.nan, dtype=np.float32)   # day x stock x feature
X[di, pj] = df[FEATS].values
Y = np.full((len(DAYS), len(PERM)), np.nan, dtype=np.float32)               # next day's excess return
Y[di, pj] = df["target_next"].values
MEM = np.zeros((len(DAYS), len(PERM)), dtype=bool)                           # S&P 500 member that day
MEM[di, pj] = df["in_sp500"].values > 0
del df
print(f"{len(DAYS):,} days x {len(PERM)} stocks x {len(FEATS)} features ({X.nbytes / 1e6:.0f} MB), "
      f"{DAYS[0].date()} to {DAYS[-1].date()}; S&P members per day: {MEM.sum(axis=1).mean():.0f} ({time.time() - t0:.0f}s)")

## 1. The model

Four parts, as in the paper (Figure 3). Shapes: $N$ stocks on a day, $T$ = 20 days, $C$ = 25 features, $H$ hidden units, $K$ factors, $M$ portfolios.

| Part | Input → output | Used in |
|---|---|---|
| **Feature extractor** | each stock's $T \times C$ history → summary $e$ ($H$ numbers), via a GRU | training and prediction |
| **Factor encoder** (teacher) | $e$ + **tomorrow's returns** → $M$ portfolio returns → posterior factors $N(\mu_{post}, \sigma_{post})$ | training only |
| **Factor predictor** (student) | $e$ → $K$ attention heads over all stocks → prior factors $N(\mu_{prior}, \sigma_{prior})$ | training and prediction |
| **Factor decoder** | $e$ + factors → each stock's return $\sim N(\alpha + \beta z,\ \sigma^2)$ | training and prediction |

**Loss** = how badly the teacher's factors explain tomorrow's returns (negative log-likelihood) + $\gamma$ × how far the student's factors are from the teacher's (KL divergence).

In [ ]:
class FactorVAE(nn.Module):
    def __init__(self, C, H=32, K=6, M=24):
        super().__init__()
        self.proj, self.gru = nn.Linear(C, H), nn.GRU(H, H, batch_first=True)            # feature extractor
        self.port, self.map_mu, self.map_sd = nn.Linear(H, M), nn.Linear(M, K), nn.Linear(M, K)   # encoder
        self.alpha_h, self.alpha_mu, self.alpha_sd = nn.Linear(H, H), nn.Linear(H, 1), nn.Linear(H, 1)  # decoder
        self.beta = nn.Linear(H, K)
        self.q = nn.Parameter(torch.randn(K, H) * 0.1)                                    # predictor: one query per head
        self.wk, self.wv = nn.Linear(H, H, bias=False), nn.Linear(H, H, bias=False)
        self.prior_h, self.prior_mu, self.prior_sd = nn.Linear(H, H), nn.Linear(H, 1), nn.Linear(H, 1)

    def features(self, x):                          # x: N x T x C -> e: N x H (GRU's last hidden state)
        _, h = self.gru(Fn.leaky_relu(self.proj(x)))
        return h[-1]

    def encoder(self, e, y):                        # teacher: posterior factors from tomorrow's returns y
        a = torch.softmax(self.port(e), dim=0)      # N x M portfolio weights (each portfolio sums to 1)
        yp = a.T @ y                                # M portfolio returns
        return self.map_mu(yp), Fn.softplus(self.map_sd(yp)) + 1e-4

    def predictor(self, e):                         # student: prior factors from the past only
        k, v = self.wk(e), self.wv(e)
        sim = Fn.relu(Fn.normalize(self.q, dim=1) @ Fn.normalize(k, dim=1).T)   # K x N attention scores
        h = Fn.leaky_relu(self.prior_h((sim / (sim.sum(dim=1, keepdim=True) + 1e-8)) @ v))
        return self.prior_mu(h).squeeze(-1), Fn.softplus(self.prior_sd(h)).squeeze(-1) + 1e-4

    def decoder(self, e, mu_z, sd_z):               # each stock's return ~ N(alpha + beta z, sd^2)
        h = Fn.leaky_relu(self.alpha_h(e))
        mu_a, sd_a = self.alpha_mu(h).squeeze(-1), Fn.softplus(self.alpha_sd(h)).squeeze(-1) + 1e-4
        b = self.beta(e)
        return mu_a + b @ mu_z, torch.sqrt(sd_a ** 2 + (b ** 2) @ (sd_z ** 2))

    def loss(self, x, y):
        e = self.features(x)
        mu_post, sd_post = self.encoder(e, y)
        mu_rec, sd_rec = self.decoder(e, mu_post, sd_post)
        nll = (torch.log(sd_rec) + 0.5 * ((y - mu_rec) / sd_rec) ** 2).mean()
        mu_pr, sd_pr = self.predictor(e)
        kl = (torch.log(sd_pr / sd_post) + (sd_post ** 2 + (mu_post - mu_pr) ** 2) / (2 * sd_pr ** 2) - 0.5).sum()
        return nll + GAMMA * kl

    @torch.no_grad()
    def predict(self, x):                           # prediction: student + decoder only (no future data)
        e = self.features(x)
        return self.decoder(e, *self.predictor(e))

print(f"Parameters: {sum(p.numel() for p in FactorVAE(len(FEATS), H, K, M_PORT).parameters()):,}")

## 2. Walk-forward training

For each predicted year:
1. **Training days:** up to the past 5 years. The last training day is dropped, because its target (the next day's return) falls in the predicted year.
2. **Standardize** each feature with the training days' mean and standard deviation (S&P members only), clip at ±5 (fat tails), and set missing values to 0 (= average).
3. **Train** one day (one cross-section of about 500 stocks) at a time. After each pass over the training days, check the **rank IC on the last year of the window**; keep the best version and stop after 2 passes without improvement.
4. **Predict** every day of the year: mean $\mu$ and standard deviation $\sigma$ for every member stock.

The prediction made at day $t$'s close is stored under day $t+1$ (the day it predicts), the same convention as the GDBN notebooks.

In [ ]:
def batch(t, idx, mu, sd):
    x = (X[t - T_DAYS + 1: t + 1, idx] - mu) / sd                  # T x N x C
    x = np.clip(np.nan_to_num(x, nan=0.0), -5, 5).transpose(1, 0, 2)
    return torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)).to(DEVICE)


def zscore(y):
    return np.clip((y - y.mean()) / (y.std() + 1e-8), -5, 5).astype(np.float32)


def rank_corr(a, b):
    ra, rb = pd.Series(a).rank().values, pd.Series(b).rank().values
    return np.corrcoef(ra, rb)[0, 1]


def fit_year(year):
    first = DAYS.searchsorted(pd.Timestamp(f"{year}-01-01"))
    last = DAYS.searchsorted(pd.Timestamp(f"{year + 1}-01-01")) - 1
    start = max(T_DAYS, DAYS.searchsorted(pd.Timestamp(f"{year - WINDOW_YEARS}-01-01")))
    window = np.arange(start, first - 1)                           # target of day first-1 is in the predicted year
    train_days, val_days = window[:-VAL_DAYS], window[-VAL_DAYS:]
    sl = X[window][MEM[window]]                                    # member rows in the window (for z-scores)
    mu, sd = np.nanmean(sl, axis=0), np.nanstd(sl, axis=0) + 1e-6

    def members(t, need_y=True):
        ok = MEM[t] & ~np.isnan(X[t, :, 0])
        return np.where(ok & ~np.isnan(Y[t]))[0] if need_y else np.where(ok)[0]

    torch.manual_seed(SEED + year)
    rng = np.random.default_rng(SEED + year)
    model = FactorVAE(len(FEATS), H, K, M_PORT).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    best, best_state, bad, epochs = -np.inf, None, 0, 0
    for epoch in range(MAX_EPOCHS):
        model.train()
        for t in rng.permutation(train_days):
            idx = members(t)
            loss = model.loss(batch(t, idx, mu, sd), torch.from_numpy(zscore(Y[t, idx])).to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        val_ic = np.nanmean([rank_corr(model.predict(batch(t, idx, mu, sd))[0].numpy(), Y[t, idx])
                             for t in val_days for idx in [members(t)]])
        epochs += 1
        if val_ic > best:
            best, best_state, bad = val_ic, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    model.eval()

    rows = []
    for t in range(first, min(last, len(DAYS) - 2) + 1):          # predict day t+1 from day t's close
        idx = members(t, need_y=False)
        m_, s_ = model.predict(batch(t, idx, mu, sd))
        rows.append(pd.DataFrame({"date": DAYS[t + 1], "permno": PERM[idx], "mu": m_.numpy(), "sigma": s_.numpy()}))
    return pd.concat(rows), {"year": year, "train days": len(train_days), "epochs": epochs, "best val IC": best}


if SCORES_FILE.exists() and not RETRAIN:
    preds = pd.read_csv(SCORES_FILE, parse_dates=["date"])
    log = pd.read_csv(OUT_DIR / "factorvae_training_log.csv")
    print(f"Loaded saved predictions: {len(preds):,} stock-days ({SCORES_FILE.name}). Set RETRAIN = True to train again.")
else:
    parts, log = [], []
    for year in YEARS:
        t1 = time.time()
        p, info = fit_year(year)
        parts.append(p); log.append(info)
        print(f"  {year}: {info['train days']} training days, {info['epochs']} passes, "
              f"best validation IC {info['best val IC']:.4f} ({time.time() - t1:.0f}s; total {(time.time() - t0) / 60:.0f} min)")
    preds, log = pd.concat(parts), pd.DataFrame(log)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    preds.to_csv(SCORES_FILE, index=False, compression="gzip")
    log.to_csv(OUT_DIR / "factorvae_training_log.csv", index=False)
    print(f"Saved predictions to {SCORES_FILE}")
log

## 3. Benchmarks on the same universe

- **Linear model:** a ridge regression of the (standardized) next-day return on the same 25 features (today's values only), re-fit every year on the same windows. This is the "linear factor model": if FactorVAE can't beat it, the deep model adds nothing.
- **Daily reversal** (−today's excess return), **weekly reversal** (−past week's), **12-month momentum**: single-feature rules.

All signals are indexed by the day they predict and use only S&P members on the decision day.

In [ ]:
def to_frame(values_by_t):
    # day-t values (decision day) -> score table indexed by day t+1 (the day predicted); members only;
    # same years as FactorVAE (2006 on), so every signal is compared over the same days
    f = pd.DataFrame(np.where(MEM, values_by_t, np.nan), index=DAYS, columns=PERM).shift(1)
    return f.reindex(index=D.days, columns=D.RET.columns).loc[f"{YEARS[0]}-01-01":pc.EVAL_END].dropna(how="all")


def linear_scores():
    out = np.full(Y.shape, np.nan, dtype=np.float32)
    for year in YEARS:
        first = DAYS.searchsorted(pd.Timestamp(f"{year}-01-01"))
        last = DAYS.searchsorted(pd.Timestamp(f"{year + 1}-01-01")) - 1
        start = DAYS.searchsorted(pd.Timestamp(f"{year - WINDOW_YEARS}-01-01"))
        window = np.arange(start, first - 1)
        sl = X[window][MEM[window]]
        mu, sd = np.nanmean(sl, axis=0), np.nanstd(sl, axis=0) + 1e-6
        Xs, ys = [], []
        for t in window[::2]:                                          # every other day (enough data, faster)
            idx = np.where(MEM[t] & ~np.isnan(Y[t]) & ~np.isnan(X[t, :, 0]))[0]
            Xs.append(np.clip(np.nan_to_num((X[t, idx] - mu) / sd), -5, 5)); ys.append(zscore(Y[t, idx]))
        model = Ridge(alpha=10.0).fit(np.vstack(Xs), np.concatenate(ys))
        for t in range(first, last + 1):
            out[t] = model.predict(np.clip(np.nan_to_num((X[t] - mu) / sd), -5, 5))
    return out


mu_tab = preds.pivot(index="date", columns="permno", values="mu").reindex(columns=D.RET.columns)
sd_tab = preds.pivot(index="date", columns="permno", values="sigma").reindex(columns=D.RET.columns)
j = {f: i for i, f in enumerate(FEATS)}
signals = {
    "FactorVAE": mu_tab,
    "Linear model (same 25 features)": to_frame(linear_scores()),
    "Daily reversal": to_frame(-X[:, :, j["exret_1d"]]),
    "Weekly reversal": to_frame(-X[:, :, j["exret_5d"]]),
    "12-month momentum": to_frame(X[:, :, j["mom_12_1"]]),
}
print({n: f"{s.notna().sum(axis=1).mean():.0f} stocks/day" for n, s in signals.items()})

## 4. Using the risk estimate (the paper's "TDrisk")

FactorVAE predicts a bell curve for each stock. The paper ranks by **mean − η × standard deviation**, so uncertain stocks are penalized. η is chosen by the best rank IC on **2006–2020 only**.

In [ ]:
target = pc.forward_excess(D, mu_tab, 1)
etas = [0.0, 0.02, 0.05, 0.1, 0.2, 0.5]
eta_ic = {e: pc.rank_ic(mu_tab - e * sd_tab, target).loc[:pc.TUNE_END].mean() for e in etas}
ETA = max(eta_ic, key=eta_ic.get)
print("2006-2020 rank IC by η:", {e: round(v, 4) for e, v in eta_ic.items()}, f"-> η = {ETA}")
signals["FactorVAE (mean − η·sd)"] = mu_tab - ETA * sd_tab

## 5. Results (all ~500 S&P 500 members)

Same tools as the GDBN notebooks: **rank IC** (mean, Newey–West t, and ICIR = mean / std), and a daily **long–short** portfolio (buy the top 20%, short the bottom 20%, about 100 stocks each side; 5 bps cost per unit traded).

In [ ]:
def evaluate(score, a, b):
    s = score.loc[a:b]
    ic = pc.rank_ic(s, pc.forward_excess(D, s, 1))
    ls = pc.summary(D, pc.daily_portfolio(s), a, b)
    return pd.concat([pd.Series({"IC": ic.mean(), "IC t": pc.nw_t(ic), "ICIR": ic.mean() / ic.std()}), ls])


tables = {}
for period, (a, b) in pc.PERIODS.items():
    tables[period] = pd.DataFrame({n: evaluate(s, a, b) for n, s in signals.items()}).T
    print(f"===== {period} =====")
    print(tables[period].round(3).to_string(), "\n")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for name, s in signals.items():
    w = pc.daily_portfolio(s)
    g = (w * D.RET.reindex(index=w.index, columns=w.columns).fillna(0)).sum(axis=1)
    axes[0].plot((1 + g).cumprod(), label=name)
    axes[1].plot(pc.rank_ic(s, pc.forward_excess(D, s, 1)).rolling(126).mean(), label=name)
for ax in axes:
    ax.axvline(pd.Timestamp("2021-01-01"), color="gray", ls="--", lw=1); ax.legend(fontsize=7)
axes[0].set_yscale("log"); axes[0].set_title("Long–short, growth of $1 (before costs; dashed = validation)")
axes[1].axhline(0, color="black", lw=0.8); axes[1].set_title("Rank IC, 6-month rolling average")
plt.tight_layout(); plt.show()

## 6. Head-to-head with the GDBN on its own universe

The GDBN ranks the **52-stock** point-in-time universe (5 largest per sector). Here the best GDBN (notebook 10: slow inputs, next-day target) is re-run, and FactorVAE (trained on all ~500 stocks) is scored **on exactly the same stock-days**. Takes a few minutes.

In [ ]:
gd = pc.walk_forward(D, pc.slow_features, 1, label="GDBN (slow inputs)")
k_gd, _ = pc.choose_lambda(D, gd, 1)
gdbn = pc.scores(gd, k_gd)
common = gdbn.notna() & mu_tab.reindex_like(gdbn).notna()
small = {"GDBN (slow inputs)": gdbn.where(common), "FactorVAE": mu_tab.reindex_like(gdbn).where(common),
         "Weekly reversal": signals["Weekly reversal"].reindex_like(gdbn).where(common)}
tables_small = {}
for period, (a, b) in pc.PERIODS.items():
    tables_small[period] = pd.DataFrame({n: evaluate(s, a, b) for n, s in small.items()}).T
    print(f"===== {period} (GDBN universe, {common.sum(axis=1)[common.any(axis=1)].mean():.0f} stocks/day) =====")
    print(tables_small[period].round(3).to_string(), "\n")

**Reading it:**
- **FactorVAE vs the linear model:** does the deep, probabilistic model add anything beyond a straight weighted sum of the same 25 features?
- **FactorVAE vs reversal:** reversal is the strongest single feature; the model must beat it to be worth its complexity.
- **mean − η·sd:** does penalizing uncertain stocks help (as in the paper)?
- **GDBN universe:** a fair head-to-head with my GDBN on the same stocks and days.
- **2021–2022:** the only clean check (λ, η, and all settings were fixed before it).
- The paper's rank IC (0.055) came from Chinese small caps; on large US stocks, 0.01–0.03 would already be good.

## 7. Save results

In [ ]:
pd.concat(tables, names=["period", "model"]).to_csv(OUT_DIR / "factorvae_results.csv")
pd.concat(tables_small, names=["period", "model"]).to_csv(OUT_DIR / "factorvae_vs_gdbn.csv")
print("Saved to", OUT_DIR)